# Get To Know A Dataset - Spatial Atlas of Human Anatomy (SAHA)

This notebook is a guided tour of the Spatial Atlas of Human Anatomy (SAHA) v1 release. More usage examples, tutorials and documentation are in this repository (`tutorials/`, `docs/`).

**SAHA** is a multi-institutional, multi-platform spatial omics atlas of the human abdomen. The v1 release holds **126 acquisitions and 12.4 million cells from 115 donor cores (86 patients)**, measured with **CosMx** (RNA, 1K and 6K panels; 64-plex protein) and **Xenium** (377-plex, 5K and colon panels). It covers ten organs: appendix, bone marrow, colon, ileum, liver, lymph node, pancreas, prostate, rectum and stomach. Per-organ and per-panel counts are in [RELEASE_BREAKDOWN.md](docs/RELEASE_BREAKDOWN.md).

This notebook provides a **101-level introduction** to the SAHA dataset: how it is organized, what data formats are present, and how to load and visualize the data.

**Where the data are.** Until the AWS Open Data bucket (`saha-open-data`) is live, the release is served read-only from a public reviewer copy, `s3://saha-public` on Wasabi (no credentials needed). One variable, `DATA_ROOT`, in the setup cell below selects the source.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The release is organized into three public prefixes:

1. **`metadata/`** — Four Parquet tables (`samples`, `donors`, `runs`, `panels`) plus `measurements.csv` (one row per acquisition x donor core) and `cohort_coverage.csv`. These are the starting point for any analysis.
2. **`processed/<record>/<sample_id>.zarr`** — one SpatialData store per acquisition (cell table, cell polygons, transcripts for RNA, and morphology/H&E images). `<record>` is the Zenodo record slug, e.g. `saha-v1-colon`; the same objects are in the Zenodo records as `<sample_id>.zarr.zip`.
3. **`docs/`** — data dictionary, file formats, quick start and the release breakdown.

Raw instrument output is not part of the public reviewer copy.

Full documentation: [QUICK_START.md](docs/QUICK_START.md) | [DATA_DICTIONARY.md](docs/DATA_DICTIONARY.md)

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# pandas >= 2.0
# pyarrow >= 14.0
# s3fs >= 2024.2
# zarr >= 3.0
# anndata >= 0.11
# matplotlib >= 3.8
# scanpy >= 1.10
# squidpy >= 1.4
# seaborn >= 0.13

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import numpy as np

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

import os

# One setting picks the data source; every path below is built from DATA_ROOT.
#   "s3://saha-public"     reviewer copy on Wasabi: public read, no credentials (default)
#   "s3://saha-open-data"  AWS Open Data bucket, once it is live
#   "/path/to/saha"        a local copy with the same layout: metadata/, processed/<record>/
DATA_ROOT = os.environ.get("SAHA_DATA_ROOT", "s3://saha-public")
if DATA_ROOT.startswith("s3://saha-public"):
    S3_OPTS = {"anon": True, "client_kwargs": {"endpoint_url": "https://s3.wasabisys.com"}}
elif DATA_ROOT.startswith("s3://"):
    S3_OPTS = {"anon": True}
else:
    S3_OPTS = {}
META = f"{DATA_ROOT}/metadata"

import fsspec, zarr
import anndata as ad

def object_path(sample_id):
    """The store for one acquisition: processed/<record>/<sample_id>.zarr.
    (25 Xenium stores carry an _RNA suffix, so both names are accepted.)"""
    fs, root = fsspec.core.url_to_fs(DATA_ROOT, **S3_OPTS)
    hits = [h for h in fs.glob(f"{root}/processed/*/{sample_id}*.zarr")
            if h.rsplit("/", 1)[-1] in (f"{sample_id}.zarr", f"{sample_id}_RNA.zarr")]
    assert len(hits) == 1, f"{sample_id}: {hits}"
    return fs.unstrip_protocol(hits[0]) if DATA_ROOT.startswith("s3://") else hits[0]

def read_table(sample_id):
    """The cell table (an AnnData) of one object, read straight from the store.
    Only the table is fetched; images, polygons and transcripts stay where they are."""
    g = zarr.open_group(object_path(sample_id), mode="r", storage_options=S3_OPTS or None)
    adata = ad.io.read_elem(g["tables/table"])
    # 25 Xenium tables keep scaled values in X and the counts in layers["raw"];
    # put counts in X so every object starts from the same place.
    if "raw" in adata.layers:
        adata.X = adata.layers["raw"]
    return adata

# Cell types: `cell_type_broad` is the one vocabulary shared by every object.
LABEL = "cell_type_broad"

# For the full SpatialData object (images, polygons, transcripts), copy the store
# first; spatialdata does not open a remote store reliably. For example:
#   aws s3 sync --no-sign-request --endpoint-url https://s3.wasabisys.com \
#       s3://saha-public/processed/<record>/<store>.zarr ./<store>.zarr
#   sdata = spatialdata.read_zarr("./<store>.zarr")


Next, we will connect to the data source and explore its top-level structure.

In [ ]:
fs, root = fsspec.core.url_to_fs(DATA_ROOT, **S3_OPTS)

# Top-level prefixes
for p in fs.ls(root, detail=False):
    print(p)

The `processed/` prefix holds one folder per release record (the Zenodo record slug), each with one SpatialData store per acquisition.

In [ ]:
# Records under processed/, with the number of objects in each
for rec in fs.ls(f"{root}/processed", detail=False):
    print(f"  {rec.rsplit('/', 1)[-1]:28s} {len(fs.ls(rec, detail=False)):3d} objects")

In [ ]:
# Inside one store: a SpatialData zarr with tables, shapes, points and images
store = object_path("SAHA_CR_COL_A1_TAP01")
print(store)
for p in fs.ls(store, detail=False):
    print("  ", p.rsplit("/", 1)[-1])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

**Metadata (Parquet / CSV)**
The four metadata tables (`samples`, `donors`, `runs`, `panels`) are [Apache Parquet](https://parquet.apache.org/) files; `measurements.csv` and `cohort_coverage.csv` are plain CSV. They are small and read directly with pandas, R arrow or DuckDB.

**Processed objects (SpatialData / OME-NGFF Zarr)**
Each acquisition is one [SpatialData](https://spatialdata.scverse.org/) store (zarr v3). It holds the cell table (`tables/table`, an AnnData with counts or protein intensities, `obs['donor_core']`, `obs['cell_type_broad']` and `obsm['spatial']`), cell polygons, transcript locations for RNA, and the morphology and H&E images as OME-NGFF pyramids. The table can be read on its own without fetching the images (see `read_table` above); to work with the images, copy the store locally and open it with `spatialdata.read_zarr`.

Recommended tools: Python (`pandas`, `anndata`, `scanpy`, `squidpy`, `spatialdata`), R (`arrow`, `anndataR`).

### Q: Can you show us an example of downloading and loading data from your dataset?

The best starting point is the **metadata tables** — they let you discover samples, then jump directly to the underlying data. Let's load all four metadata tables and explore.

In [ ]:
samples = pd.read_parquet(f"{META}/samples.parquet", storage_options=S3_OPTS or None)
donors  = pd.read_parquet(f"{META}/donors.parquet",  storage_options=S3_OPTS or None)
runs    = pd.read_parquet(f"{META}/runs.parquet",    storage_options=S3_OPTS or None)
panels  = pd.read_parquet(f"{META}/panels.parquet",  storage_options=S3_OPTS or None)

print(f"samples: {samples.shape[0]:,} rows × {samples.shape[1]} columns")
print(f"donors:  {donors.shape[0]:,} rows × {donors.shape[1]} columns")
print(f"runs:    {runs.shape[0]:,} rows × {runs.shape[1]} columns")
print(f"panels:  {panels.shape[0]:,} rows × {panels.shape[1]} columns")

In [ ]:
# Peek at the samples table
samples[["sample_id", "organ", "platform", "condition",
         "panel_plex", "n_cells", "n_fovs", "n_donors"]].head(5)

In [ ]:
# Peek at the donors table
donors.head(5)

In [ ]:
# Panel configurations
for _, row in panels.iterrows():
    gene_count = len(row["gene_list"]) if row["gene_list"] is not None else 0
    print(f"{row['panel_name']:25s}  {row['platform']:8s}  plex={row['plex']:<6}  genes={gene_count}")

### Q: A picture is worth a thousand words. Show us visuals from your dataset that illustrate something informative or exciting.

Let's start with an overview of the atlas composition — how many samples and cells are available across organs, platforms, and conditions.

In [ ]:
# All released samples for the overview plots
qc = samples.copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=100)

# --- Samples by organ ---
organ_counts = qc["organ"].value_counts().sort_values()
organ_counts.plot(kind="barh", ax=axes[0], color="#3498db", edgecolor="white")
axes[0].set_xlabel("Number of samples")
axes[0].set_title("Samples by organ", fontweight="bold")
axes[0].spines[["top", "right"]].set_visible(False)

# --- Total cells by platform ---
cells_by_platform = qc.groupby("platform")["n_cells"].sum().sort_values()
cells_by_platform.plot(kind="barh", ax=axes[1], color="#2ecc71", edgecolor="white")
axes[1].set_xlabel("Total cells")
axes[1].set_title("Total cells by platform", fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

# --- Condition breakdown ---
cond_counts = qc["condition"].value_counts()
axes[2].pie(cond_counts, labels=cond_counts.index, autopct="%1.0f%%",
            startangle=90, colors=sns.color_palette("Set2", len(cond_counts)))
axes[2].set_title("Condition distribution", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nTotal samples:           {len(qc):,}")
print(f"Total cells:             {qc['n_cells'].sum():,.0f}")
print(f"Donor cores in release:  {int(donors['in_release'].sum()):,}")
print(f"Platforms:               {', '.join(sorted(qc['platform'].unique()))}")
print(f"Organs:                  {', '.join(sorted(qc['organ'].unique()))}")

In [ ]:
# Distribution of cell counts per sample, colored by platform
fig, ax = plt.subplots(figsize=(10, 5), dpi=100, facecolor="white")

for platform, group in qc.groupby("platform"):
    ax.hist(group["n_cells"], bins=40, alpha=0.6, label=platform, edgecolor="white")

ax.set_xlabel("Cells per sample", fontsize=12)
ax.set_ylabel("Number of samples", fontsize=12)
ax.set_title("Distribution of Cell Counts per Sample", fontsize=14, fontweight="bold")
ax.legend(title="Platform")
ax.spines[["top", "right"]].set_visible(False)
ax.set_facecolor("#f8f9fa")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Median cells/sample: {qc['n_cells'].median():,.0f}")
print(f"Max cells/sample:    {qc['n_cells'].max():,.0f}")

In [ ]:
# Heatmap: total cells per organ × platform
pivot = (
    qc.groupby(["organ", "platform"])["n_cells"]
    .sum()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
sns.heatmap(
    pivot, annot=True, fmt=",.0f", cmap="YlGnBu",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Total cells"}
)
ax.set_title("Total Cells: Organ × Platform", fontsize=14, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()
plt.show()

Now let's load a single processed sample and visualize the spatial cell type map — this is what makes SAHA a *spatial* atlas.

In [ ]:
# Pick a colon CosMx RNA sample (the release has colon on the CosMx 1K panel).
# The smallest one keeps the download to a few MB.
candidates = qc.query(
    "organ == 'colon' and platform == 'cosmx' and assay_type == 'rna'"
).sort_values("n_cells")

row = candidates.iloc[0]
print(f"Selected: {row['sample_id']}")
print(f"  organ:   {row['organ']}")
print(f"  n_cells: {row['n_cells']:,}")
print(f"  n_fovs:  {row['n_fovs']}")
print(f"  panel:   {row['panel_name']} ({row['panel_plex']}-plex)")
print(f"  store:   {object_path(row['sample_id'])}")

In [ ]:
# The release ships SpatialData (OME-NGFF / Zarr). Read only the cell table here.
adata = read_table(row["sample_id"])
print(adata)
print(f"\nCell metadata columns: {adata.obs.columns.tolist()}")
print(f"Embeddings available:  {list(adata.obsm.keys())}")

In [ ]:
# Spatial cell type map — the hallmark of spatial omics
# (mapping a categorical column to RGB tuples by hand fails in recent pandas, so let scanpy color it)
if "spatial" in adata.obsm:
    fig, ax = plt.subplots(figsize=(12, 9), dpi=100)
    sc.pl.embedding(adata, basis="spatial", color=LABEL, size=2, ax=ax, show=False, frameon=False,
                    title=f"{row['sample_id']} — cell types in tissue")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in this sample.")

In [ ]:
# Spatial expression of marker genes
marker_genes = [g for g in ["EPCAM", "CD3E", "CD68", "MKI67", "COL1A1"]
                if g in adata.var_names]

if marker_genes and "spatial" in adata.obsm:
    sq.pl.spatial_scatter(
        adata,
        color=marker_genes,
        shape=None,
        size=2,
        img=False,
        ncols=len(marker_genes),
        figsize=(4 * len(marker_genes), 4),
    )
else:
    print(f"Available marker genes: {marker_genes}")

In [ ]:
# UMAP colored by cell type (computed here when the object does not carry one)
if "X_umap" not in adata.obsm:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.pca(adata, n_comps=30)
    sc.pp.neighbors(adata, n_neighbors=15)
    sc.tl.umap(adata)
sc.pl.umap(
    adata,
    color=LABEL,
    legend_loc="on data",
    legend_fontsize=8,
    title=f"{row['sample_id']} — UMAP",
    frameon=False,
)

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question:** *How does cell type composition vary across GI organs, and do multi-organ donors show consistent immune representation?*

SAHA profiles multiple organs from the same donor, enabling direct within-donor comparisons of immune cell proportions. Below we compute cell type fractions per organ and look for patterns.

In [ ]:
# Aggregate by organ and platform. Donor counts come from n_donors -- a slide
# is a tissue microarray of up to 28 cores, so there is no donor_id per sample.
summary = (
    qc.groupby(["organ", "platform"])
    .agg(
        n_samples=("sample_id", "count"),
        total_cells=("n_cells", "sum"),
        donor_cores=("n_donors", "sum"),
    )
    .reset_index()
    .sort_values("total_cells", ascending=False)
)

print("Organ x Platform summary:")
summary.head(15)


In [ ]:
# Donors profiled across multiple organs. This has to come from donors.parquet:
# samples carries no donor column, because one acquisition images many donors.
organs_per_donor = (
    donors[donors["in_release"]]
    .groupby("donor_patient")["organ"].nunique()
    .sort_values(ascending=False)
)

multi_organ_donors = organs_per_donor[organs_per_donor > 1]
print(f"Donors with multi-organ profiling: {len(multi_organ_donors)}")
print(f"\nTop donors by number of organs profiled:")
print(multi_organ_donors.head(10))


In [ ]:
# Cells by donor CORE. Attributing a whole acquisition's cells to one donor
# would multiply-count every core on a 28-core slide, so go through
# measurements.csv, which is one row per acquisition x core.
# Donor age group, sex and ethnicity are registered-tier (ACCESS_TIERS.md), so
# the open tables break cells down by organ and condition instead.
meas = pd.read_csv(f"{META}/measurements.csv", storage_options=S3_OPTS or None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=100)

meas.groupby("condition")["cells"].sum().plot(
    kind="bar", ax=axes[0], color="#3498db", edgecolor="white"
)
axes[0].set_title("Total cells by core condition")
axes[0].set_ylabel("Cells")

meas.groupby("organ")["cells"].sum().sort_values().plot(
    kind="barh", ax=axes[1], color="#9b59b6", edgecolor="white"
)
axes[1].set_title("Total cells by organ")
axes[1].set_xlabel("Cells")

plt.tight_layout(); plt.show()


### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations for someone wanting to answer this question?

**Question:** *Can cross-platform spatial transcriptomics data be integrated to build a unified cell atlas that is robust to technological batch effects?*

SAHA profiles the same organs using multiple spatial platforms (CosMx and Xenium). This creates a unique opportunity to benchmark integration methods and build a platform-agnostic reference atlas of the human abdomen. Key challenges include:

1. **Gene panel overlap**: Different platforms measure different gene sets (CosMx 1K and 6K; Xenium 377-plex and 5K). Integration must handle the partially-shared feature space.
2. **Resolution differences**: CosMx and Xenium both resolve single cells, but segmentation, detection efficiency and cell size differ between them.
3. **Spatial alignment**: When the same tissue block is profiled on multiple platforms, spatial coordinates can be registered to enable direct pixel-level comparison.

**Recommendations:**
- Start with Tutorial 03 (`tutorials/03_cross_platform_comparison.ipynb`) which compares CosMx and Xenium on the same ileum tissue microarray.
- Use `scvi-tools` or `scanorama` for integration, benchmarking against the shared gene set.
- The `panels` metadata table provides the full gene list per panel, which is essential for identifying the intersection.
- Consider the [Open Problems in Single-Cell Analysis](https://openproblems.bio/) benchmarking framework for rigorous evaluation.

---

## Further Resources

| Resource | Description |
|----------|-------------|
| [QUICK_START.md](docs/QUICK_START.md) | Concise access patterns (Python, R, AWS CLI, Athena) |
| [DATA_DICTIONARY.md](docs/DATA_DICTIONARY.md) | Full column definitions for all metadata tables |
| [FILE_FORMATS.md](docs/FILE_FORMATS.md) | Raw data formats per platform |
| [ACCESS_TIERS.md](docs/ACCESS_TIERS.md) | How to request registered/controlled-access data |
| Tutorial 01 | [Query metadata tables](tutorials/01_query_metadata.ipynb) |
| Tutorial 02 | [Load and explore a CosMx sample](tutorials/02_load_cosmx_sample.ipynb) |
| Tutorial 03 | [Cross-platform comparison](tutorials/03_cross_platform_comparison.ipynb) |

---
*Remember to clear all outputs before committing to your GitHub repository.*